In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

DataFrame[]

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# Gold 노트북 - 지역별 최신 환경 지표 통합 테이블
# ============================================================

# ① Silver 테이블 읽기
df_weather = spark.table("silver.weather")
df_congest = spark.table("silver.congest")
df_road = spark.table("silver.road")
df_event = spark.table("silver.event_accident")

# ② 각 테이블에서 지역별 최신 데이터 1개만 추출
window_weather = Window.partitionBy("AREA_NM").orderBy(F.col("WEATHER_TIME").desc())
window_congest = Window.partitionBy("AREA_NM").orderBy(F.col("PPLTN_TIME").desc())
window_road = Window.partitionBy("AREA_NM").orderBy(F.col("ROAD_TRAFFIC_TIME").desc())
window_event = Window.partitionBy("AREA_NM").orderBy(F.col("collected_at").desc())

df_weather_latest = df_weather.withColumn("rn", F.row_number().over(window_weather)) \
    .filter(F.col("rn") == 1).drop("rn")

df_congest_latest = df_congest.withColumn("rn", F.row_number().over(window_congest)) \
    .filter(F.col("rn") == 1).drop("rn")

df_road_latest = df_road.withColumn("rn", F.row_number().over(window_road)) \
    .filter(F.col("rn") == 1).drop("rn")

df_event_latest = df_event.withColumn("rn", F.row_number().over(window_event)) \
    .filter(F.col("rn") == 1).drop("rn")

# ③ AREA_NM 기준으로 조인
df_gold = df_weather_latest.select(
    "AREA_NM", "TEMP", "SENSIBLE_TEMP", "HUMIDITY",
    "WIND_SPD", "WIND_DIRCT", "PRECIPITATION", "PRECPT_TYPE",
    "PCP_MSG", "UV_INDEX", "UV_INDEX_LVL",
    "PM10", "PM10_INDEX", "PM25", "PM25_INDEX",
    "AIR_IDX", "AIR_MSG", "WEATHER_TIME"
).join(
    df_congest_latest.select(
        "AREA_NM", "AREA_CONGEST_LVL", "AREA_CONGEST_MSG",
        "AREA_PPLTN_MIN", "AREA_PPLTN_MAX", "PPLTN_TIME"
    ), on="AREA_NM", how="left"
).join(
    df_road_latest.select(
        "AREA_NM", "ROAD_TRAFFIC_SPD", "ROAD_TRAFFIC_IDX", "ROAD_TRAFFIC_TIME"
    ), on="AREA_NM", how="left"
).join(
    df_event_latest.select(
        "AREA_NM", "ACDNT_CNT", "EVENT_YN", "DST_MSG_CNT"
    ), on="AREA_NM", how="left"
)

# ④ Gold Delta Table로 저장
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.walk_environment")

print("✅ Gold 저장 완료! 🐾")

✅ Gold 저장 완료! 🐾


In [0]:
spark.sql("SELECT * FROM gold.walk_environment").display()

AREA_NM,TEMP,SENSIBLE_TEMP,HUMIDITY,WIND_SPD,WIND_DIRCT,PRECIPITATION,PRECPT_TYPE,PCP_MSG,UV_INDEX,UV_INDEX_LVL,PM10,PM10_INDEX,PM25,PM25_INDEX,AIR_IDX,AIR_MSG,WEATHER_TIME,AREA_CONGEST_LVL,AREA_CONGEST_MSG,AREA_PPLTN_MIN,AREA_PPLTN_MAX,PPLTN_TIME,ROAD_TRAFFIC_SPD,ROAD_TRAFFIC_IDX,ROAD_TRAFFIC_TIME,ACDNT_CNT,EVENT_YN,DST_MSG_CNT
가로수길,9.9,10.5,95,2.7,NNE,1.5mm,비,약한 비가 내리고 있어요.외출 시 우산을 챙기세요.,낮음,1,17,좋음,13,좋음,보통,호흡기가 예민하신 분들은 몸상태 유의하며 활동해주세요.,2026-04-09 16:00,보통,사람이 몰려있을 수 있지만 크게 붐비지는 않아요. 도보 이동에 큰 제약이 없어요.,32000,34000,2026-04-09 15:40,17,서행,2026-04-09 16:05,0,0,0
강남 MICE 관광특구,10.0,10.0,96,1.1,ENE,1.5mm,비,약한 비가 내리고 있어요.외출 시 우산을 챙기세요.,낮음,1,17,좋음,13,좋음,보통,호흡기가 예민하신 분들은 몸상태 유의하며 활동해주세요.,2026-04-09 16:00,약간 붐빔,사람들이 몰려있을 가능성이 크고 붐빈다고 느낄 수 있어요. 인구밀도가 높은 구간에서는 도보 이동시 부딪힘이 발생할 수 있어요.,28000,30000,2026-04-09 15:40,12,정체,2026-04-09 16:05,0,0,0
강남역,10.0,10.0,96,1.1,ENE,1.5mm,비,약한 비가 내리고 있어요.외출 시 우산을 챙기세요.,낮음,1,26,좋음,23,보통,보통,호흡기가 예민하신 분들은 몸상태 유의하며 활동해주세요.,2026-04-09 16:00,약간 붐빔,사람들이 몰려있을 가능성이 크고 붐빈다고 느낄 수 있어요. 인구밀도가 높은 구간에서는 도보 이동시 부딪힘이 발생할 수 있어요.,86000,88000,2026-04-09 15:40,12,정체,2026-04-09 16:05,0,0,0
고덕역,9.4,9.8,95,3.1,NE,1.0mm,비,약한 비가 내리고 있어요.외출 시 우산을 챙기세요.,낮음,1,16,좋음,16,보통,보통,호흡기가 예민하신 분들은 몸상태 유의하며 활동해주세요.,2026-04-09 16:00,여유,사람이 몰려있을 가능성이 낮고 붐빔은 거의 느껴지지 않아요. 도보 이동이 자유로워요.,2500,3000,2026-04-09 15:40,21,서행,2026-04-09 16:05,0,0,0
고속터미널역,9.8,11.3,98,1.4,NNE,1.5mm,비,약한 비가 내리고 있어요.외출 시 우산을 챙기세요.,낮음,1,26,좋음,23,보통,보통,호흡기가 예민하신 분들은 몸상태 유의하며 활동해주세요.,2026-04-09 16:00,약간 붐빔,사람들이 몰려있을 가능성이 크고 붐빈다고 느낄 수 있어요. 인구밀도가 높은 구간에서는 도보 이동시 부딪힘이 발생할 수 있어요.,20000,22000,2026-04-09 15:40,16,서행,2026-04-09 16:05,0,0,0
교대역,10.0,10.0,96,1.1,ENE,1.5mm,비,약한 비가 내리고 있어요.외출 시 우산을 챙기세요.,낮음,1,26,좋음,23,보통,보통,호흡기가 예민하신 분들은 몸상태 유의하며 활동해주세요.,2026-04-09 16:00,붐빔,사람들이 몰려있을 가능성이 매우 크고 많이 붐빈다고 느낄 수 있어요. 인구밀도가 높은 구간에서는 도보 이동시 부딪힘이 발생할 수 있어요.,24000,26000,2026-04-09 15:40,15,서행,2026-04-09 16:05,0,0,0
서울 암사동 유적,9.5,10.5,96,2.1,NNE,1.5mm,비,약한 비가 내리고 있어요.외출 시 우산을 챙기세요.,낮음,1,16,좋음,16,보통,보통,호흡기가 예민하신 분들은 몸상태 유의하며 활동해주세요.,2026-04-09 16:00,여유,사람이 몰려있을 가능성이 낮고 붐빔은 거의 느껴지지 않아요. 도보 이동이 자유로워요.,200,300,2026-04-09 15:40,23,서행,2026-04-09 16:05,0,0,0
선릉역,10.0,10.0,96,1.1,ENE,1.5mm,비,약한 비가 내리고 있어요.외출 시 우산을 챙기세요.,낮음,1,17,좋음,13,좋음,보통,호흡기가 예민하신 분들은 몸상태 유의하며 활동해주세요.,2026-04-09 16:00,약간 붐빔,사람들이 몰려있을 가능성이 크고 붐빈다고 느낄 수 있어요. 인구밀도가 높은 구간에서는 도보 이동시 부딪힘이 발생할 수 있어요.,66000,68000,2026-04-09 15:40,14,정체,2026-04-09 16:05,0,0,0
송리단길·호수단길,10.3,10.3,94,1.3,ESE,2.0mm,비,약한 비가 내리고 있어요.외출 시 우산을 챙기세요.,낮음,1,15,좋음,14,좋음,보통,호흡기가 예민하신 분들은 몸상태 유의하며 활동해주세요.,2026-04-09 16:00,여유,사람이 몰려있을 가능성이 낮고 붐빔은 거의 느껴지지 않아요. 도보 이동이 자유로워요.,6000,6500,2026-04-09 15:40,17,서행,2026-04-09 16:05,0,0,0
신논현역·논현역,10.0,10.0,96,1.1,ENE,1.5mm,비,약한 비가 내리고 있어요.외출 시 우산을 챙기세요.,낮음,1,17,좋음,13,좋음,보통,호흡기가 예민하신 분들은 몸상태 유의하며 활동해주세요.,2026-04-09 16:00,보통,사람이 몰려있을 수 있지만 크게 붐비지는 않아요. 도보 이동에 큰 제약이 없어요.,14000,16000,2026-04-09 15:40,13,정체,2026-04-09 16:05,0,0,0


In [0]:
# Databricks 노트북에서 실행
display(spark.sql("SELECT * FROM gold.walk_environment LIMIT 5"))

AREA_NM,TEMP,SENSIBLE_TEMP,HUMIDITY,WIND_SPD,WIND_DIRCT,PRECIPITATION,PRECPT_TYPE,PCP_MSG,UV_INDEX,UV_INDEX_LVL,PM10,PM10_INDEX,PM25,PM25_INDEX,AIR_IDX,AIR_MSG,WEATHER_TIME,AREA_CONGEST_LVL,AREA_CONGEST_MSG,AREA_PPLTN_MIN,AREA_PPLTN_MAX,PPLTN_TIME,ROAD_TRAFFIC_SPD,ROAD_TRAFFIC_IDX,ROAD_TRAFFIC_TIME,ACDNT_CNT,EVENT_YN,DST_MSG_CNT
가로수길,9.9,10.5,95,2.7,NNE,1.5mm,비,약한 비가 내리고 있어요.외출 시 우산을 챙기세요.,낮음,1,17,좋음,13,좋음,보통,호흡기가 예민하신 분들은 몸상태 유의하며 활동해주세요.,2026-04-09 16:00,보통,사람이 몰려있을 수 있지만 크게 붐비지는 않아요. 도보 이동에 큰 제약이 없어요.,32000,34000,2026-04-09 15:40,17,서행,2026-04-09 16:05,0,0,0
강남 MICE 관광특구,10.0,10.0,96,1.1,ENE,1.5mm,비,약한 비가 내리고 있어요.외출 시 우산을 챙기세요.,낮음,1,17,좋음,13,좋음,보통,호흡기가 예민하신 분들은 몸상태 유의하며 활동해주세요.,2026-04-09 16:00,약간 붐빔,사람들이 몰려있을 가능성이 크고 붐빈다고 느낄 수 있어요. 인구밀도가 높은 구간에서는 도보 이동시 부딪힘이 발생할 수 있어요.,28000,30000,2026-04-09 15:40,12,정체,2026-04-09 16:05,0,0,0
강남역,10.0,10.0,96,1.1,ENE,1.5mm,비,약한 비가 내리고 있어요.외출 시 우산을 챙기세요.,낮음,1,26,좋음,23,보통,보통,호흡기가 예민하신 분들은 몸상태 유의하며 활동해주세요.,2026-04-09 16:00,약간 붐빔,사람들이 몰려있을 가능성이 크고 붐빈다고 느낄 수 있어요. 인구밀도가 높은 구간에서는 도보 이동시 부딪힘이 발생할 수 있어요.,86000,88000,2026-04-09 15:40,12,정체,2026-04-09 16:05,0,0,0
고덕역,9.4,9.8,95,3.1,NE,1.0mm,비,약한 비가 내리고 있어요.외출 시 우산을 챙기세요.,낮음,1,16,좋음,16,보통,보통,호흡기가 예민하신 분들은 몸상태 유의하며 활동해주세요.,2026-04-09 16:00,여유,사람이 몰려있을 가능성이 낮고 붐빔은 거의 느껴지지 않아요. 도보 이동이 자유로워요.,2500,3000,2026-04-09 15:40,21,서행,2026-04-09 16:05,0,0,0
고속터미널역,9.8,11.3,98,1.4,NNE,1.5mm,비,약한 비가 내리고 있어요.외출 시 우산을 챙기세요.,낮음,1,26,좋음,23,보통,보통,호흡기가 예민하신 분들은 몸상태 유의하며 활동해주세요.,2026-04-09 16:00,약간 붐빔,사람들이 몰려있을 가능성이 크고 붐빈다고 느낄 수 있어요. 인구밀도가 높은 구간에서는 도보 이동시 부딪힘이 발생할 수 있어요.,20000,22000,2026-04-09 15:40,16,서행,2026-04-09 16:05,0,0,0
